# MelodyLang Compiler

A compiler for **MelodyLang**, a small domain-specific language for music, that turns text instructions into a MIDI file.

**Pipeline:** Lexer &rarr; Parser (AST) &rarr; Semantic Analyzer &rarr; Pattern Compressor (AST optimization) &rarr; IR Generator (musical intermediate representation) &rarr; IR Optimizer (merges repeated note/rest events) &rarr; Code Generator (MIDI output).

Run the cells top to bottom.

In [28]:
!pip install midiutil pretty_midi -q



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. The compiler (lexer, parser, semantic analysis, IR, optimizer, codegen)

In [29]:
"""
MelodyLang Compiler
====================
A compiler for a small domain-specific music language (MelodyLang) that
converts text-based music instructions into MIDI output.

Pipeline:
    source text
        -> Lexer            (tokenizes)
        -> Parser            (builds an AST)
        -> SemanticAnalyzer   (validates pitches, durations, ranges)
        -> PatternCompressor  (AST-level optimization: folds repeated
                                statement runs into RepeatNodes)
        -> IRGenerator        (lowers AST into a flat musical IR:
                                a timeline of NoteEvent / RestEvent)
        -> IROptimizer        (merges consecutive rests and tied notes)
        -> CodeGenerator      (IR -> a .mid file via midiutil)

Example MelodyLang program:

    tempo 120
    instrument piano

    note C4 quarter
    note E4 quarter
    note G4 half

    chord [C4, E4, G4] whole

    rest quarter

    repeat 4 {
        note C4 eighth
        note D4 eighth
    }
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import List, Optional, Union


# ---------------------------------------------------------------------------
# 1. LEXER
# ---------------------------------------------------------------------------

TOKEN_SPEC = [
    ("SKIP", r"[ \t\r\n]+"),
    ("COMMENT", r"//[^\n]*"),
    ("LBRACE", r"\{"),
    ("RBRACE", r"\}"),
    ("LBRACKET", r"\["),
    ("RBRACKET", r"\]"),
    ("COMMA", r","),
    ("INT", r"\d+"),
    ("IDENT", r"[A-Za-z][A-Za-z0-9#\-]*"),
]
MASTER_RE = re.compile("|".join(f"(?P<{name}>{pat})" for name, pat in TOKEN_SPEC))


@dataclass
class Token:
    type: str
    value: str
    line: int


class MelodyLangSyntaxError(Exception):
    pass


class Lexer:
    def __init__(self, source: str):
        self.source = source

    def tokenize(self) -> List[Token]:
        tokens = []
        line = 1
        pos = 0
        length = len(self.source)
        while pos < length:
            m = MASTER_RE.match(self.source, pos)
            if not m:
                bad = self.source[pos]
                raise MelodyLangSyntaxError(f"Line {line}: unexpected character {bad!r}")
            kind = m.lastgroup
            text = m.group()
            if kind == "SKIP":
                line += text.count("\n")
            elif kind == "COMMENT":
                pass
            else:
                tokens.append(Token(kind, text, line))
            pos = m.end()
        tokens.append(Token("EOF", "", line))
        return tokens


# ---------------------------------------------------------------------------
# 2. AST NODES
# ---------------------------------------------------------------------------

class Node:
    pass


@dataclass(eq=True)
class TempoNode(Node):
    bpm: int
    line: int = field(compare=False, default=0)


@dataclass(eq=True)
class InstrumentNode(Node):
    name: str
    line: int = field(compare=False, default=0)


@dataclass(eq=True)
class NoteNode(Node):
    pitch: str
    duration: str
    line: int = field(compare=False, default=0)


@dataclass(eq=True)
class ChordNode(Node):
    pitches: List[str]
    duration: str
    line: int = field(compare=False, default=0)


@dataclass(eq=True)
class RestNode(Node):
    duration: str
    line: int = field(compare=False, default=0)


@dataclass(eq=True)
class RepeatNode(Node):
    count: int
    body: List[Node]
    line: int = field(compare=False, default=0)


# ---------------------------------------------------------------------------
# 3. PARSER (recursive descent)
# ---------------------------------------------------------------------------

class Parser:
    def __init__(self, tokens: List[Token]):
        self.tokens = tokens
        self.pos = 0

    def peek(self) -> Token:
        return self.tokens[self.pos]

    def advance(self) -> Token:
        tok = self.tokens[self.pos]
        if tok.type != "EOF":
            self.pos += 1
        return tok

    def expect(self, type_: str) -> Token:
        tok = self.peek()
        if tok.type != type_:
            raise MelodyLangSyntaxError(
                f"Line {tok.line}: expected {type_} but found {tok.type} ({tok.value!r})"
            )
        return self.advance()

    def expect_ident(self, text: str) -> Token:
        tok = self.peek()
        if tok.type != "IDENT" or tok.value.lower() != text:
            raise MelodyLangSyntaxError(
                f"Line {tok.line}: expected keyword '{text}' but found {tok.value!r}"
            )
        return self.advance()

    def parse_program(self) -> List[Node]:
        statements = []
        while self.peek().type != "EOF":
            statements.append(self.parse_statement())
        return statements

    def parse_statement(self) -> Node:
        tok = self.peek()
        if tok.type != "IDENT":
            raise MelodyLangSyntaxError(f"Line {tok.line}: expected a statement, found {tok.value!r}")
        keyword = tok.value.lower()
        if keyword == "tempo":
            return self.parse_tempo()
        elif keyword == "instrument":
            return self.parse_instrument()
        elif keyword == "note":
            return self.parse_note()
        elif keyword == "chord":
            return self.parse_chord()
        elif keyword == "rest":
            return self.parse_rest()
        elif keyword == "repeat":
            return self.parse_repeat()
        else:
            raise MelodyLangSyntaxError(f"Line {tok.line}: unknown statement '{tok.value}'")

    def parse_tempo(self) -> TempoNode:
        line = self.peek().line
        self.expect_ident("tempo")
        bpm_tok = self.expect("INT")
        return TempoNode(bpm=int(bpm_tok.value), line=line)

    def parse_instrument(self) -> InstrumentNode:
        line = self.peek().line
        self.expect_ident("instrument")
        name_tok = self.expect("IDENT")
        return InstrumentNode(name=name_tok.value.lower(), line=line)

    def parse_note(self) -> NoteNode:
        line = self.peek().line
        self.expect_ident("note")
        pitch_tok = self.expect("IDENT")
        dur_tok = self.expect("IDENT")
        return NoteNode(pitch=pitch_tok.value, duration=dur_tok.value.lower(), line=line)

    def parse_chord(self) -> ChordNode:
        line = self.peek().line
        self.expect_ident("chord")
        self.expect("LBRACKET")
        pitches = [self.expect("IDENT").value]
        while self.peek().type == "COMMA":
            self.advance()
            pitches.append(self.expect("IDENT").value)
        self.expect("RBRACKET")
        dur_tok = self.expect("IDENT")
        return ChordNode(pitches=pitches, duration=dur_tok.value.lower(), line=line)

    def parse_rest(self) -> RestNode:
        line = self.peek().line
        self.expect_ident("rest")
        dur_tok = self.expect("IDENT")
        return RestNode(duration=dur_tok.value.lower(), line=line)

    def parse_repeat(self) -> RepeatNode:
        line = self.peek().line
        self.expect_ident("repeat")
        count_tok = self.expect("INT")
        self.expect("LBRACE")
        body = []
        while self.peek().type != "RBRACE":
            if self.peek().type == "EOF":
                raise MelodyLangSyntaxError(f"Line {line}: unterminated repeat block")
            body.append(self.parse_statement())
        self.expect("RBRACE")
        return RepeatNode(count=int(count_tok.value), body=body, line=line)


# ---------------------------------------------------------------------------
# 4. SEMANTIC ANALYSIS
# ---------------------------------------------------------------------------

PITCH_RE = re.compile(r"^([A-Ga-g])(#|b)?(-?\d+)$")

DURATIONS = {
    "whole": 4.0, "half": 2.0, "quarter": 1.0, "eighth": 0.5, "sixteenth": 0.25,
    "dotted-whole": 6.0, "dotted-half": 3.0, "dotted-quarter": 1.5,
    "dotted-eighth": 0.75, "dotted-sixteenth": 0.375,
}

GM_INSTRUMENTS = {
    "piano": 0, "acoustic-piano": 0, "electric-piano": 4, "harpsichord": 6,
    "guitar": 24, "acoustic-guitar": 24, "electric-guitar": 27,
    "bass": 33, "violin": 40, "viola": 41, "cello": 42, "strings": 48,
    "trumpet": 56, "trombone": 57, "saxophone": 65, "flute": 73,
    "clarinet": 71, "organ": 19, "synth": 80, "drums": -1,
}


class MelodyLangSemanticError(Exception):
    def __init__(self, errors: List[str]):
        self.errors = errors
        super().__init__("\n".join(errors))


class SemanticAnalyzer:
    """Validates the AST: pitch names, octave ranges, durations,
    instrument names, tempo range, and repeat counts."""

    MIN_OCTAVE, MAX_OCTAVE = 0, 8
    MIN_TEMPO, MAX_TEMPO = 20, 300
    MAX_REPEAT = 1000

    def __init__(self):
        self.errors: List[str] = []

    def analyze(self, program: List[Node]) -> List[Node]:
        self._walk(program)
        if self.errors:
            raise MelodyLangSemanticError(self.errors)
        return program

    def _walk(self, statements: List[Node]):
        for stmt in statements:
            if isinstance(stmt, TempoNode):
                if not (self.MIN_TEMPO <= stmt.bpm <= self.MAX_TEMPO):
                    self.errors.append(
                        f"Line {stmt.line}: tempo {stmt.bpm} out of range "
                        f"({self.MIN_TEMPO}-{self.MAX_TEMPO} BPM)"
                    )
            elif isinstance(stmt, InstrumentNode):
                if stmt.name not in GM_INSTRUMENTS:
                    self.errors.append(
                        f"Line {stmt.line}: unknown instrument '{stmt.name}'. "
                        f"Valid: {', '.join(sorted(GM_INSTRUMENTS))}"
                    )
            elif isinstance(stmt, NoteNode):
                self._check_pitch(stmt.pitch, stmt.line)
                self._check_duration(stmt.duration, stmt.line)
            elif isinstance(stmt, ChordNode):
                if not stmt.pitches:
                    self.errors.append(f"Line {stmt.line}: chord has no notes")
                for p in stmt.pitches:
                    self._check_pitch(p, stmt.line)
                self._check_duration(stmt.duration, stmt.line)
            elif isinstance(stmt, RestNode):
                self._check_duration(stmt.duration, stmt.line)
            elif isinstance(stmt, RepeatNode):
                if not (1 <= stmt.count <= self.MAX_REPEAT):
                    self.errors.append(
                        f"Line {stmt.line}: repeat count {stmt.count} out of range "
                        f"(1-{self.MAX_REPEAT})"
                    )
                self._walk(stmt.body)
            else:
                self.errors.append(f"Unknown AST node: {stmt!r}")

    def _check_pitch(self, pitch: str, line: int):
        m = PITCH_RE.match(pitch)
        if not m:
            self.errors.append(f"Line {line}: invalid pitch '{pitch}' (expected e.g. C4, F#3, Bb5)")
            return
        octave = int(m.group(3))
        if not (self.MIN_OCTAVE <= octave <= self.MAX_OCTAVE):
            self.errors.append(f"Line {line}: octave in '{pitch}' out of range (0-8)")

    def _check_duration(self, duration: str, line: int):
        if duration not in DURATIONS:
            self.errors.append(
                f"Line {line}: invalid duration '{duration}'. Valid: {', '.join(DURATIONS)}"
            )


# ---------------------------------------------------------------------------
# 5. AST-LEVEL OPTIMIZATION: repeated-pattern compression
# ---------------------------------------------------------------------------

class PatternCompressor:
    """Detects contiguous runs of repeated statement sequences and folds
    them into RepeatNodes, shrinking the AST before lowering to IR.
    e.g. [note C4 q, note C4 q, note C4 q] -> [repeat 3 { note C4 q }]
    """

    def __init__(self):
        self.folds_applied = 0

    def compress(self, statements: List[Node]) -> List[Node]:
        statements = [self._compress_children(s) for s in statements]
        return self._compress_flat(statements)

    def _compress_children(self, stmt: Node) -> Node:
        if isinstance(stmt, RepeatNode):
            stmt.body = self.compress(stmt.body)
        return stmt

    def _compress_flat(self, stmts: List[Node]) -> List[Node]:
        result = []
        i = 0
        n = len(stmts)
        while i < n:
            best_len, best_reps = 0, 1
            max_pattern_len = (n - i) // 2
            for plen in range(1, max_pattern_len + 1):
                pattern = stmts[i:i + plen]
                reps = 1
                j = i + plen
                while j + plen <= n and stmts[j:j + plen] == pattern:
                    reps += 1
                    j += plen
                if reps >= 2 and plen * reps > best_len * best_reps:
                    best_len, best_reps = plen, reps
            if best_reps >= 2:
                pattern = stmts[i:i + best_len]
                result.append(RepeatNode(count=best_reps, body=pattern,
                                          line=pattern[0].line if hasattr(pattern[0], "line") else 0))
                self.folds_applied += 1
                i += best_len * best_reps
            else:
                result.append(stmts[i])
                i += 1
        return result


# ---------------------------------------------------------------------------
# 6. MUSICAL INTERMEDIATE REPRESENTATION (IR)
# ---------------------------------------------------------------------------

@dataclass
class MusicEvent:
    kind: str            # "note" or "rest"
    start_beat: float
    duration_beats: float
    pitches: List[int] = field(default_factory=list)  # MIDI note numbers (empty for rest)
    velocity: int = 100


def pitch_to_midi(pitch: str) -> int:
    m = PITCH_RE.match(pitch)
    letter, accidental, octave = m.group(1).upper(), m.group(2), int(m.group(3))
    base = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}[letter]
    if accidental == "#":
        base += 1
    elif accidental == "b":
        base -= 1
    return (octave + 1) * 12 + base


@dataclass
class IRResult:
    events: List[MusicEvent]
    tempo: int
    instrument: str


class IRGenerator:
    """Lowers the (compressed) AST into a flat timeline of MusicEvents,
    expanding repeat loops as it walks."""

    def __init__(self):
        self.events: List[MusicEvent] = []
        self.cursor = 0.0
        self.tempo = 120
        self.instrument = "piano"

    def generate(self, program: List[Node]) -> IRResult:
        self._walk(program)
        return IRResult(events=self.events, tempo=self.tempo, instrument=self.instrument)

    def _walk(self, statements: List[Node]):
        for stmt in statements:
            if isinstance(stmt, TempoNode):
                self.tempo = stmt.bpm
            elif isinstance(stmt, InstrumentNode):
                self.instrument = stmt.name
            elif isinstance(stmt, NoteNode):
                dur = DURATIONS[stmt.duration]
                self.events.append(MusicEvent("note", self.cursor, dur, [pitch_to_midi(stmt.pitch)]))
                self.cursor += dur
            elif isinstance(stmt, ChordNode):
                dur = DURATIONS[stmt.duration]
                self.events.append(MusicEvent("note", self.cursor, dur,
                                               [pitch_to_midi(p) for p in stmt.pitches]))
                self.cursor += dur
            elif isinstance(stmt, RestNode):
                dur = DURATIONS[stmt.duration]
                self.events.append(MusicEvent("rest", self.cursor, dur))
                self.cursor += dur
            elif isinstance(stmt, RepeatNode):
                for _ in range(stmt.count):
                    self._walk(stmt.body)


# ---------------------------------------------------------------------------
# 7. IR OPTIMIZER: merges repeated / adjacent note & rest events
# ---------------------------------------------------------------------------

class IROptimizer:
    """Two peephole passes over the flat event timeline:
       1. merge consecutive rests into a single longer rest
       2. tie consecutive notes of identical pitch set into one note
    """

    def optimize(self, events: List[MusicEvent]) -> (List[MusicEvent], dict):
        original_count = len(events)
        merged = []
        for ev in events:
            if merged:
                prev = merged[-1]
                contiguous = abs((prev.start_beat + prev.duration_beats) - ev.start_beat) < 1e-9
                if contiguous and prev.kind == "rest" and ev.kind == "rest":
                    prev.duration_beats += ev.duration_beats
                    continue
                if contiguous and prev.kind == "note" and ev.kind == "note" and prev.pitches == ev.pitches:
                    prev.duration_beats += ev.duration_beats
                    continue
            merged.append(MusicEvent(ev.kind, ev.start_beat, ev.duration_beats,
                                      list(ev.pitches), ev.velocity))
        stats = {
            "events_before": original_count,
            "events_after": len(merged),
            "events_merged": original_count - len(merged),
        }
        return merged, stats


# ---------------------------------------------------------------------------
# 8. CODE GENERATION: IR -> MIDI file
# ---------------------------------------------------------------------------

class CodeGenerator:
    def generate(self, ir: IRResult, output_path: str):
        from midiutil import MIDIFile

        program = GM_INSTRUMENTS.get(ir.instrument, 0)
        channel = 9 if ir.instrument == "drums" else 0
        midi = MIDIFile(1)
        track = 0
        midi.addTrackName(track, 0, "MelodyLang")
        midi.addTempo(track, 0, ir.tempo)
        if program >= 0:
            midi.addProgramChange(track, channel, 0, program)

        for ev in ir.events:
            if ev.kind == "note":
                for pitch in ev.pitches:
                    midi.addNote(track, channel, pitch, ev.start_beat, ev.duration_beats, ev.velocity)
            # rests need no MIDI event; they are represented by the gap in time

        with open(output_path, "wb") as f:
            midi.writeFile(f)


# ---------------------------------------------------------------------------
# 9. TOP-LEVEL COMPILER
# ---------------------------------------------------------------------------

class MelodyLangCompiler:
    def compile_to_file(self, source: str, output_path: str) -> dict:
        tokens = Lexer(source).tokenize()
        ast = Parser(tokens).parse_program()
        SemanticAnalyzer().analyze(ast)

        compressor = PatternCompressor()
        compressed_ast = compressor.compress(ast)

        ir_result = IRGenerator().generate(compressed_ast)
        optimized_events, opt_stats = IROptimizer().optimize(ir_result.events)
        ir_result.events = optimized_events

        CodeGenerator().generate(ir_result, output_path)

        return {
            "output_file": output_path,
            "tempo": ir_result.tempo,
            "instrument": ir_result.instrument,
            "tokens": len(tokens),
            "top_level_statements": len(ast),
            "pattern_folds_applied": compressor.folds_applied,
            **opt_stats,
        }


# ---------------------------------------------------------------------------
# 10. DEMO PROGRAM
# ---------------------------------------------------------------------------

SAMPLE_PROGRAM = """
tempo 120
instrument piano

note C4 quarter
note E4 quarter
note G4 half

chord [C4, E4, G4] whole

rest quarter
rest quarter

repeat 4 {
    note C4 eighth
    note D4 eighth
}

note C5 eighth
note D5 eighth
note C5 eighth
note D5 eighth
note C5 eighth
note D5 eighth
"""


## 2. Sample MelodyLang program

```
tempo 120
instrument piano

note C4 quarter
note E4 quarter
note G4 half

chord [C4, E4, G4] whole

rest quarter
rest quarter

repeat 4 {
    note C4 eighth
    note D4 eighth
}
```

In [30]:
compiler = MelodyLangCompiler()
report = compiler.compile_to_file(SAMPLE_PROGRAM, "melody_output.mid")

print("Compilation report:")
for k, v in report.items():
    print(f"  {k}: {v}")


Compilation report:
  output_file: melody_output.mid
  tempo: 120
  instrument: piano
  tokens: 55
  top_level_statements: 15
  pattern_folds_applied: 2
  events_before: 20
  events_after: 19
  events_merged: 1


## 3. Listen to the result
This renders the MIDI file to audio directly in the notebook (uses a built-in FluidSynth soundfont via `pretty_midi`).

In [31]:
import os
import subprocess
from IPython.display import Audio, display

import pretty_midi

midi_path = "melody_output.mid"
soundfont_path = "/usr/share/sounds/sf2/FluidR3_GM.sf2"

if os.name == "nt":
    print(f"Created {midi_path}. Audio preview is unavailable in this Windows cell.")
    print("Open the MIDI file in a MIDI player, MuseScore, or VLC to listen to it.")
else:
    # FluidSynth + a General MIDI soundfont are needed to render audio from MIDI
    if not os.path.exists(soundfont_path):
        subprocess.run(["apt-get", "install", "-y", "-qq", "fluidsynth", "fluid-soundfont-gm"], check=True)

    pm = pretty_midi.PrettyMIDI(midi_path)
    audio_data = pm.fluidsynth(fs=44100, sf2_path=soundfont_path)
    display(Audio(audio_data, rate=44100))

Created melody_output.mid. Audio preview is unavailable in this Windows cell.
Open the MIDI file in a MIDI player, MuseScore, or VLC to listen to it.


## 4. Download the MIDI file

In [32]:
from pathlib import Path
from IPython.display import FileLink, display

midi_path = Path("melody_output.mid")
if midi_path.exists():
    display(FileLink(str(midi_path)))
else:
    print(f"File not found: {midi_path}")

c:\Users\Sahil Chandel\Downloads\pcd\melody_output.mid

## 5. Write your own MelodyLang program

In [33]:
my_program = """
tempo 140
instrument violin

note A4 quarter
note B4 quarter
note C5 quarter
note D5 quarter

chord [A4, C5, E5] half
rest quarter

repeat 6 {
    note E5 sixteenth
    note D5 sixteenth
}
"""

report2 = compiler.compile_to_file(my_program, "my_melody.mid")
print("Compilation report:")
for k, v in report2.items():
    print(f"  {k}: {v}")

if os.name == "nt":
    print("Created my_melody.mid. Open it in a MIDI player to listen to it.")
else:
    pm2 = pretty_midi.PrettyMIDI("my_melody.mid")
    audio2 = pm2.fluidsynth(fs=44100, sf2_path="/usr/share/sounds/sf2/FluidR3_GM.sf2")
    display(Audio(audio2, rate=44100))

Compilation report:
  output_file: my_melody.mid
  tempo: 140
  instrument: violin
  tokens: 38
  top_level_statements: 9
  pattern_folds_applied: 0
  events_before: 18
  events_after: 18
  events_merged: 0
Created my_melody.mid. Open it in a MIDI player to listen to it.


## Language reference

| Statement | Syntax | Example |
|---|---|---|
| Tempo | `tempo <BPM>` | `tempo 120` |
| Instrument | `instrument <name>` | `instrument piano` |
| Note | `note <pitch> <duration>` | `note C4 quarter` |
| Chord | `chord [<pitch>, ...] <duration>` | `chord [C4, E4, G4] whole` |
| Rest | `rest <duration>` | `rest half` |
| Repeat | `repeat <n> { ... }` | `repeat 4 { note C4 eighth }` |

**Pitches:** letter A-G, optional `#`/`b`, octave 0-8 (e.g. `C4`, `F#3`, `Bb5`).
**Durations:** `whole`, `half`, `quarter`, `eighth`, `sixteenth`, or `dotted-<duration>`.
**Instruments:** piano, electric-piano, harpsichord, guitar, electric-guitar, bass, violin, viola, cello, strings, trumpet, trombone, saxophone, flute, clarinet, organ, synth, drums.

## 6. Individual compiler test cases

Run the compiler-definition cell first, then run these four test cells separately.

In [38]:
# Test 1: valid 30-second piano tune
# 10 beats per motif x 5 repeats = 50 beats.
# At 100 BPM: 50 / 100 x 60 = 30 seconds.
compiler = MelodyLangCompiler()
program = """tempo 100
instrument piano
repeat 5 {
    note C4 quarter
    note E4 quarter
    note G4 quarter
    note C5 quarter
    chord [C4, E4, G4] quarter
    note B4 quarter
    note A4 quarter
    note G4 quarter
    note E4 quarter
    note D4 quarter
}
"""

print("INPUT: 30-second piano tune")
print(program)
report = compiler.compile_to_file(program, "test_1_30sec_piano.mid")
print("OUTPUT:")
print("PASS: generated test_1_30sec_piano.mid")
print("Approximate duration: 30.0 seconds")
print(report)

INPUT: 30-second piano tune
tempo 100
instrument piano
repeat 5 {
    note C4 quarter
    note E4 quarter
    note G4 quarter
    note C5 quarter
    chord [C4, E4, G4] quarter
    note B4 quarter
    note A4 quarter
    note G4 quarter
    note E4 quarter
    note D4 quarter
}

OUTPUT:
PASS: generated test_1_30sec_piano.mid
Approximate duration: 30.0 seconds
{'output_file': 'test_1_30sec_piano.mid', 'tempo': 100, 'instrument': 'piano', 'tokens': 45, 'top_level_statements': 3, 'pattern_folds_applied': 0, 'events_before': 50, 'events_after': 50, 'events_merged': 0}


In [39]:
# Test 2: valid 30-second flute tune
# 9 beats per motif x 5 repeats = 45 beats.
# At 90 BPM: 45 / 90 x 60 = 30 seconds.
compiler = MelodyLangCompiler()
program = """tempo 90
instrument flute
repeat 5 {
    note D4 quarter
    note E4 quarter
    note F4 quarter
    note G4 quarter
    note A4 quarter
    note G4 quarter
    chord [D4, F4, A4] quarter
    chord [E4, G4, B4] quarter
    note D5 quarter
}
"""

print("INPUT: 30-second flute tune")
print(program)
report = compiler.compile_to_file(program, "test_2_30sec_flute.mid")
print("OUTPUT:")
print("PASS: generated test_2_30sec_flute.mid")
print("Approximate duration: 30.0 seconds")
print(report)

INPUT: 30-second flute tune
tempo 90
instrument flute
repeat 5 {
    note D4 quarter
    note E4 quarter
    note F4 quarter
    note G4 quarter
    note A4 quarter
    note G4 quarter
    chord [D4, F4, A4] quarter
    chord [E4, G4, B4] quarter
    note D5 quarter
}

OUTPUT:
PASS: generated test_2_30sec_flute.mid
Approximate duration: 30.0 seconds
{'output_file': 'test_2_30sec_flute.mid', 'tempo': 90, 'instrument': 'flute', 'tokens': 48, 'top_level_statements': 3, 'pattern_folds_applied': 0, 'events_before': 45, 'events_after': 45, 'events_merged': 0}


In [36]:
# Test 3: invalid syntax - the note is missing its duration
program = """tempo 120
instrument piano
note C4
"""

print("INPUT:")
print(program)
print("OUTPUT:")
try:
    tokens = Lexer(program).tokenize()
    Parser(tokens).parse_program()
    print("FAIL: the missing duration was not detected")
except MelodyLangSyntaxError as error:
    print(f"EXPECTED SYNTAX ERROR: {error}")

INPUT:
tempo 120
instrument piano
note C4

OUTPUT:
EXPECTED SYNTAX ERROR: Line 4: expected IDENT but found EOF ('')


In [37]:
# Test 4: invalid semantics - tempo is outside the allowed 20-300 range
program = """tempo 500
instrument violin
note D4 quarter
"""

print("INPUT:")
print(program)
print("OUTPUT:")
try:
    tokens = Lexer(program).tokenize()
    ast = Parser(tokens).parse_program()
    SemanticAnalyzer().analyze(ast)
    print("FAIL: the invalid tempo was not detected")
except MelodyLangSemanticError as error:
    print(f"EXPECTED SEMANTIC ERROR: {error}")

INPUT:
tempo 500
instrument violin
note D4 quarter

OUTPUT:
EXPECTED SEMANTIC ERROR: Line 1: tempo 500 out of range (20-300 BPM)
